In [0]:
df = spark.read.option("sep", ",") \
    .option("header", True) \
    .option("inferSchema",True) \
    .csv("s3n://humber-lfb-databricks-class-files/telco_customer_data.csv")

df.head(10)

[Row(CUSTOMER ID='K100010', GENDER='Male', AGE=46, POSTAL CODE=6253, REGION=3, TARIFF='CAT 50', HANDSET='SOP10', DROPPED CALLS=1, CONNECT DATE='15-Jul-16', END DATE='7-May-20'),
 Row(CUSTOMER ID='K100020', GENDER='Male', AGE=27, POSTAL CODE=4121, REGION=2, TARIFF='CAT 50', HANDSET='SOP10', DROPPED CALLS=0, CONNECT DATE='16-Sep-14', END DATE='4-Feb-17'),
 Row(CUSTOMER ID='K100030', GENDER='Male', AGE=39, POSTAL CODE=3870, REGION=2, TARIFF='CAT 50', HANDSET='SOP20', DROPPED CALLS=2, CONNECT DATE='20-Aug-13', END DATE='10-Aug-16'),
 Row(CUSTOMER ID='K100040', GENDER='Male', AGE=28, POSTAL CODE=8322, REGION=4, TARIFF='CAT 50', HANDSET='SOP10', DROPPED CALLS=2, CONNECT DATE='16-Aug-15', END DATE='1-Jun-16'),
 Row(CUSTOMER ID='K100050', GENDER='Male', AGE=47, POSTAL CODE=2614, REGION=2, TARIFF='CAT 50', HANDSET='SOP10', DROPPED CALLS=0, CONNECT DATE='9-Aug-15', END DATE='10-Aug-16'),
 Row(CUSTOMER ID='K100060', GENDER='Male', AGE=29, POSTAL CODE=1891, REGION=1, TARIFF='CAT 50', HANDSET='SOP1

In [0]:
import pandas as pd

pdf = df.toPandas()

In [0]:
pdf.columns = (
    pd.Series(df.columns)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
)

In [0]:
pdf.head(10)

,customer_id,gender,age,postal_code,region,tariff,handset,dropped_calls,connect_date,end_date
0,K100010,Male,46,6253,3,CAT 50,SOP10,1,15-Jul-16,7-May-20
1,K100020,Male,27,4121,2,CAT 50,SOP10,0,16-Sep-14,4-Feb-17
2,K100030,Male,39,3870,2,CAT 50,SOP20,2,20-Aug-13,10-Aug-16
3,K100040,Male,28,8322,4,CAT 50,SOP10,2,16-Aug-15,1-Jun-16
4,K100050,Male,47,2614,2,CAT 50,SOP10,0,9-Aug-15,10-Aug-16
5,K100060,Male,29,1891,1,CAT 50,SOP10,1,4-Jul-14,11-Mar-15
6,K100070,Male,38,1741,1,CAT 50,SOP20,1,11-Oct-16,29-Dec-16
7,K100080,Male,27,9770,4,CAT 50,SOP20,0,18-Jan-15,25-Aug-19
8,K100090,Male,38,8967,4,CAT 50,SOP20,1,28-May-13,17-May-14
9,K100100,Male,48,3438,2,CAT 50,SOP20,0,25-Jul-13,2-Jul-17


In [0]:
spark.createDataFrame(pdf).createOrReplaceTempView('telecom_subscriber_info')

In [0]:
%sql
-- Question-1: Dropped Call Analysis 
-- a. Average  number of dropped calls per postal code.
-- b. Top 10 postal codes based on the average number of dropped calls.

SELECT postal_code, ROUND(AVG(dropped_calls) ,2) AS avg_dropped_calls
FROM telecom_subscriber_info
GROUP BY postal_code
ORDER BY avg_dropped_calls DESC
LIMIT 10;


postal_code,avg_dropped_calls
6826,44.0
3169,41.0
3994,41.0
7857,24.5
2005,23.5
5500,21.0
7921,21.0
3331,20.0
2525,20.0
7055,19.0


In [0]:
%sql
-- Question-2: Tariff Distribution and Handset Analysis
-- c. Write a query to get the count of customers by TARIFF.
-- d. Visualize if desired (optional).

SELECT tariff, COUNT(customer_id) AS total_customers
FROM telecom_subscriber_info
GROUP BY tariff;

tariff,total_customers
CAT 100,6680
Play 100,5825
Play 300,3582
CAT 200,13725
CAT 50,1993


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- QUESTION 3: Handset Analysis 
-- e. Compare the count of handset models (SOP10, SOP20, etc.) across tariffs
-- f. For each tariff, determine how many customers have SOP10, SOP20, etc.

SELECT tariff, handset AS handset_model, COUNT(handset) AS total_handset_models, COUNT(customer_id) AS total_customers
FROM telecom_subscriber_info
GROUP BY tariff, handset;

tariff,handset_model,total_handset_models,total_customers
CAT 50,SOP20,87,87
Play 100,S80,955,955
CAT 200,S50,3461,3461
CAT 200,S80,1132,1132
Play 300,S50,476,476
Play 100,SOP10,180,180
CAT 50,SOP10,73,73
Play 100,SOP20,182,182
Play 300,ASAD170,382,382
Play 300,ASAD90,719,719


In [0]:
%sql

--Question 4: Recomendation analysis

--sum of dropped calls by handset and tariffs. Total call drops number is a lot more on some handsets and some tariffs than others
SELECT SUM(dropped_calls) AS total_dropped_calls, handset
FROM telecom_subscriber_info
GROUP BY handset
ORDER BY total_dropped_calls DESC;

SELECT SUM(dropped_calls) AS total_dropped_calls, tariff
FROM telecom_subscriber_info
GROUP BY tariff
ORDER BY total_dropped_calls DESC;

total_dropped_calls,tariff
43106,CAT 200
21343,CAT 100
19001,Play 100
11685,Play 300
6294,CAT 50
